# Baseline lengkap untuk Section 3.3 — SIVP

Notebook mandiri. Tidak memerlukan berkas `.py` tambahan.

**Isi**

1. Konfigurasi dan impor
2. Definisi fungsi (sel 3--8, jalankan sekali, tidak menghasilkan keluaran)
3. Muat data
4. Baseline fitur: colour moments, histogram RGB, ECDF-7 vs ECDF-9
5. Perbandingan representasi dan uji t berpasangan
6. Protokol B: cross-validation bebas kebocoran
7. Grid search baseline clustering
8. ResNet-50 (opsional, paling lama)
9. Ekspor tabel ke LaTeX

Jalankan sel berurutan dari atas. Sel 1--7 selesai sekitar 5--7 menit.

In [2]:
import os, ctypes, pathlib

shim = pathlib.Path.home() / "torch_dll_shim"
os.add_dll_directory(str(shim))
ctypes.CDLL(str(shim / "libomp140.x86_64.dll"))   # kunci OpenMP yang benar

import torch
from torch import nn
from torchvision import models, transforms
print("torch siap:", torch.__version__)

torch siap: 2.8.0+cpu


## 1. Konfigurasi

In [3]:
# Sesuaikan path berikut. Di Windows pakai garis miring biasa, bukan backslash.
EXCEL      = "C:/Users/Ahsan/Downloads/data/data_ekstraksi_rgb_4kelas.xlsx"
OUTDIR     = "hasil_baseline"

# Folder INDUK citra asli, yang berisi subfolder pink_rose, red_rose,
# white_rose, yellow_rose. Hanya dipakai untuk ResNet-50 di sel 8.
IMAGE_ROOT = "C:/Users/Ahsan/Downloads/data"

HIST_BINS = 4     # histogram RGB gabungan: 4^3 = 64 dimensi
N_SEEDS   = 100   # Protokol A

In [4]:
import ast, json, os, time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (accuracy_score, adjusted_rand_score,
                             calinski_harabasz_score, davies_bouldin_score,
                             normalized_mutual_info_score,
                             precision_recall_fscore_support, silhouette_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import (MinMaxScaler, QuantileTransformer,
                                   StandardScaler)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

N_SEEDS_CV       = 20   # Protokol B
N_SEEDS_INTERNAL = 20   # silhouette bersifat O(n^2), cukup 20 seed
LEVELS = (100, 150, 200)
COLS7  = [("R", 100), ("R", 150), ("R", 200),
          ("G", 150), ("G", 200), ("B", 150), ("B", 200)]

os.makedirs(OUTDIR, exist_ok=True)
print("siap")

c:\ProgramData\anaconda3\envs\gbp\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


siap


## 2. Definisi fungsi

Sel 3 sampai 8 hanya mendefinisikan fungsi dan tidak mencetak apa pun.
Jalankan semuanya sekali.

### Pemuatan data

In [5]:
def load_pixels(excel_path):
    """Kembalikan array citra (N, 32, 32, 3), label, dan nama berkas.

    Kolom R_values/G_values/B_values disimpan sebagai literal list Python oleh
    notebook rose_4_warna.ipynb. Jika berkas Anda menyimpannya sebagai repr
    numpy terpotong (mengandung '...'), data mentahnya hilang dan berkas harus
    diekstrak ulang dengan str(array.tolist()).
    """
    df = pd.read_excel(excel_path)
    sample = str(df['R_values'].iloc[0])
    if '...' in sample:
        raise ValueError(
            "Kolom piksel tersimpan sebagai repr numpy terpotong. "
            "Ekstrak ulang dengan str(row['R_values'].tolist()).")

    def parse(s):
        return np.asarray(ast.literal_eval(s), dtype=np.uint8)

    R = np.stack(df['R_values'].map(parse).values)
    G = np.stack(df['G_values'].map(parse).values)
    B = np.stack(df['B_values'].map(parse).values)
    p = int(df['p'].iloc[0])
    imgs = np.stack([R, G, B], axis=-1).reshape(-1, p, p, 3)
    return imgs, df['label'].values, df['filename'].values

### Ekstraktor fitur

In [6]:
def feat_ecdf7(imgs):
    """Deskriptor yang diusulkan: ECDF per kanal pada 3 level, 7 koordinat."""
    flat = imgs.reshape(len(imgs), -1, 3).astype(np.int16)
    ch = {'R': flat[:, :, 0], 'G': flat[:, :, 1], 'B': flat[:, :, 2]}
    return np.column_stack([(ch[c] <= t).mean(axis=1) for c, t in COLS7])


def feat_ecdf9(imgs):
    """Grid penuh 3 kanal x 3 level, 9 koordinat."""
    flat = imgs.reshape(len(imgs), -1, 3).astype(np.int16)
    ch = {'R': flat[:, :, 0], 'G': flat[:, :, 1], 'B': flat[:, :, 2]}
    return np.column_stack([(ch[c] <= t).mean(axis=1)
                            for c in ('R', 'G', 'B') for t in LEVELS])


def feat_colour_moments(imgs):
    """Colour moments Stricker & Orengo: mean, sd, skewness per kanal (d=9)."""
    flat = imgs.reshape(len(imgs), -1, 3).astype(np.float64)
    mean = flat.mean(axis=1)
    sd = flat.std(axis=1)
    skew = stats.skew(flat, axis=1)
    return np.column_stack([mean, sd, skew])


def feat_rgb_histogram(imgs, bins=4):
    """Histogram RGB gabungan terkuantisasi, dinormalisasi. d = bins^3.

    bins=4 memberi d=64 (dipakai di naskah). bins=8 memberi d=512; laporkan
    keduanya jika reviewer menanyakan sensitivitas terhadap kuantisasi.
    """
    flat = imgs.reshape(len(imgs), -1, 3).astype(np.int32)
    q = flat * bins // 256                       # indeks bin per kanal
    idx = q[:, :, 0] * bins * bins + q[:, :, 1] * bins + q[:, :, 2]
    d = bins ** 3
    out = np.zeros((len(imgs), d), dtype=np.float64)
    for i in range(len(imgs)):
        out[i] = np.bincount(idx[i], minlength=d)
    return out / flat.shape[1]

### ResNet-50

Hanya dipakai di sel 8.

In [7]:
def feat_resnet50(imgs=None, image_root=None, filenames=None, batch=64):
    """Fitur lapisan penultimate ResNet-50 pra-latih ImageNet (d=2048).

    Dua mode:
      image_root diberikan -> baca citra ASLI dari disk (direkomendasikan)
      image_root None      -> pakai citra 32x32 hasil rekonstruksi, di-upsample
    """
    import torch
    from torch import nn
    from torchvision import models, transforms
    from PIL import Image

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    net.fc = nn.Identity()                       # ambil keluaran 2048-d
    net.eval().to(device)

    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
    tf_disk = transforms.Compose([transforms.Resize(256),
                                  transforms.CenterCrop(224),
                                  transforms.ToTensor(), norm])

    feats = []
    with torch.no_grad():
        if image_root is not None:
            # Indeks nama berkas -> path, dibangun SEKALI. Versi sebelumnya
            # memanggil os.walk untuk setiap berkas, yang berperilaku O(n^2).
            index = {}
            collisions = set()
            for root, _, files in os.walk(image_root):
                for f in files:
                    if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        continue
                    if f in index:
                        collisions.add(f)
                    index[f] = os.path.join(root, f)
            if collisions:
                raise ValueError(
                    f'{len(collisions)} nama berkas muncul di lebih dari satu '
                    f'subfolder, misalnya {sorted(collisions)[:5]}. Pemetaan '
                    'citra ke label menjadi ambigu. Beri prefiks kelas pada '
                    'nama berkas, atau ubah fungsi ini agar memakai '
                    'subfolder sebagai kunci.')

            missing = [f for f in filenames if f not in index]
            if missing:
                raise FileNotFoundError(
                    f'{len(missing)} berkas tidak ditemukan di {image_root}, '
                    f'misalnya {missing[:5]}')
            paths = [index[f] for f in filenames]
            print(f'{len(paths)} citra terindeks dari {image_root}')

            for i in range(0, len(paths), batch):
                imgs_t = torch.stack([tf_disk(Image.open(p).convert('RGB'))
                                      for p in paths[i:i + batch]]).to(device)
                feats.append(net(imgs_t).cpu().numpy())
                if (i // batch) % 10 == 0:
                    print(f'  {min(i + batch, len(paths))}/{len(paths)}',
                          end='\r')
        else:
            x = torch.from_numpy(imgs).permute(0, 3, 1, 2).float() / 255.0
            for i in range(0, len(x), batch):
                b = torch.nn.functional.interpolate(
                    x[i:i + batch], size=224, mode='bilinear',
                    align_corners=False)
                b = torch.stack([norm(t) for t in b]).to(device)
                feats.append(net(b).cpu().numpy())
    return np.vstack(feats)

### Transformasi representasi

In [8]:
def rank_transform(X, ref=None):
    """Transformasi rank Definisi 2.5. ref=fold latih untuk Protokol B."""
    ref = X if ref is None else ref
    out = np.empty(X.shape, dtype=float)
    for k in range(X.shape[1]):
        s = np.sort(ref[:, k])
        out[:, k] = np.searchsorted(s, X[:, k], side='right') / len(s)
    return out


def make_representations(F):
    qt = QuantileTransformer(output_distribution='uniform',
                             n_quantiles=min(1000, len(F)),
                             random_state=0)
    return {
        'raw': F,
        'minmax': MinMaxScaler().fit_transform(F),
        'zscore': StandardScaler().fit_transform(F),
        'rank': rank_transform(F),
        'quantile': qt.fit_transform(F),
    }

### Metrik dan protokol evaluasi

In [9]:
def hungarian_map(y_true, cl, labs):
    """Pemetaan cluster ke kelas yang optimal (Kuhn 1955). Noise -1 diabaikan."""
    valid = cl >= 0
    if valid.sum() == 0:
        return np.array([labs[0]] * len(cl))
    cids = np.unique(cl[valid])
    cm = np.zeros((len(labs), len(cids)))
    for i, l in enumerate(labs):
        for j, c in enumerate(cids):
            cm[i, j] = np.sum((y_true == l) & (cl == c))
    r, c = linear_sum_assignment(-cm)
    m = {cids[c[i]]: labs[r[i]] for i in range(len(r))}
    return np.array([m.get(x, '__noise__') for x in cl])


def external_metrics(y, yp, cl):
    p, r, f, _ = precision_recall_fscore_support(
        y, yp, average='weighted', zero_division=0)
    return dict(acc=accuracy_score(y, yp) * 100, prec=p * 100, rec=r * 100,
                f1=f * 100, ari=adjusted_rand_score(y, cl),
                nmi=normalized_mutual_info_score(y, cl))


def internal_metrics(X, cl):
    if len(np.unique(cl[cl >= 0])) < 2:
        return dict(sil=np.nan, db=np.nan, ch=np.nan)
    m = cl >= 0
    return dict(sil=silhouette_score(X[m], cl[m]),
                db=davies_bouldin_score(X[m], cl[m]),
                ch=calinski_harabasz_score(X[m], cl[m]))

In [10]:
def protocol_A(X, y, labs, K, n_seeds=N_SEEDS, tag=''):
    """Protokol A: 100 seed k-means++ pada seluruh data."""
    rows = []
    t0 = time.time()
    for seed in range(n_seeds):
        cl = KMeans(K, init='k-means++', n_init=1,
                    random_state=seed).fit_predict(X)
        rec = external_metrics(y, hungarian_map(y, cl, labs), cl)
        if seed < N_SEEDS_INTERNAL:
            rec.update(internal_metrics(X, cl))
        rows.append(rec)
    df = pd.DataFrame(rows)
    out = {'representation': tag, 'n_seeds': n_seeds,
           'runtime_s': round(time.time() - t0, 2)}
    for c in df.columns:
        v = df[c].dropna().values
        out[f'{c}_mean'] = v.mean()
        out[f'{c}_sd'] = v.std(ddof=1)
    a = df['acc'].values
    out['acc_min'], out['acc_max'] = a.min(), a.max()
    out['acc_ci_lo'], out['acc_ci_hi'] = np.percentile(a, [2.5, 97.5])
    return out, df['acc'].values


def protocol_B(F, y, labs, K, transform='raw', n_seeds=N_SEEDS_CV):
    """Protokol B: 5-fold, transformasi di-fit HANYA pada fold latih."""
    skf = StratifiedKFold(5, shuffle=True, random_state=0)
    accs = []
    for seed in range(n_seeds):
        for tr, te in skf.split(F, y):
            if transform == 'raw':
                Xtr, Xte = F[tr], F[te]
            elif transform == 'minmax':
                sc = MinMaxScaler().fit(F[tr])
                Xtr, Xte = sc.transform(F[tr]), sc.transform(F[te])
            elif transform == 'zscore':
                sc = StandardScaler().fit(F[tr])
                Xtr, Xte = sc.transform(F[tr]), sc.transform(F[te])
            elif transform == 'rank':
                Xtr, Xte = rank_transform(F[tr]), rank_transform(F[te], ref=F[tr])
            elif transform == 'quantile':
                qt = QuantileTransformer(output_distribution='uniform',
                                         n_quantiles=min(1000, len(tr)),
                                         random_state=0).fit(F[tr])
                Xtr, Xte = qt.transform(F[tr]), qt.transform(F[te])
            km = KMeans(K, init='k-means++', n_init=1, random_state=seed).fit(Xtr)
            cl = km.predict(Xte)
            accs.append(accuracy_score(
                y[te], hungarian_map(y[te], cl, labs)) * 100)
    a = np.array(accs)
    return dict(transform=transform, n_runs=len(a), acc_mean=a.mean(),
                acc_sd=a.std(ddof=1),
                acc_ci_lo=np.percentile(a, 2.5),
                acc_ci_hi=np.percentile(a, 97.5))

### Grid search baseline clustering

In [11]:
def grid_search_baselines(X, y, labs, K, n_seeds=30):
    """Grid search dengan kriteria seleksi BEBAS LABEL (silhouette).

    Memakai akurasi untuk memilih hyperparameter akan membocorkan label dan
    membuat perbandingan tidak adil. Silhouette dipilih karena tersedia untuk
    keempat algoritma. Konfigurasi terpilih lalu dievaluasi terhadap label.
    """
    results = []

    # --- K-Means (metode yang diusulkan) ---
    accs, sils, t0 = [], [], time.time()
    for seed in range(n_seeds):
        cl = KMeans(K, init='k-means++', n_init=1, random_state=seed).fit_predict(X)
        accs.append(accuracy_score(y, hungarian_map(y, cl, labs)) * 100)
        sils.append(silhouette_score(X, cl))
    results.append(dict(method='K-Means (k-means++)', config=f'K={K}',
                        acc_mean=np.mean(accs), acc_sd=np.std(accs, ddof=1),
                        sil=np.mean(sils), runtime_s=(time.time() - t0) / n_seeds))

    # --- Gaussian Mixture: grid pada covariance_type dan inisialisasi ---
    # init_params bawaan sklearn adalah 'kmeans' dengan random_state yang sama,
    # sehingga GMM konvergen ke partisi yang identik dengan K-Means dan
    # perbandingannya menjadi tidak informatif. Kami menguji kedua inisialisasi
    # dan melaporkan keduanya.
    for init in ['kmeans', 'random_from_data']:
        best = None
        for cov in ['full', 'tied', 'diag', 'spherical']:
            s = []
            for seed in range(10):
                cl = GaussianMixture(K, covariance_type=cov, init_params=init,
                                     random_state=seed, n_init=1).fit_predict(X)
                s.append(silhouette_score(X, cl))
            if best is None or np.mean(s) > best[1]:
                best = (cov, np.mean(s))
        cov = best[0]
        accs, sils, t0 = [], [], time.time()
        for seed in range(n_seeds):
            cl = GaussianMixture(K, covariance_type=cov, init_params=init,
                                 random_state=seed, n_init=1).fit_predict(X)
            accs.append(accuracy_score(y, hungarian_map(y, cl, labs)) * 100)
            sils.append(silhouette_score(X, cl))
        results.append(dict(
            method=f'Gaussian mixture ({init})', config=f'cov={cov}',
            acc_mean=np.mean(accs), acc_sd=np.std(accs, ddof=1),
            sil=np.mean(sils), runtime_s=(time.time() - t0) / n_seeds))

    # --- Agglomerative: grid pada linkage (deterministik, SD = 0) ---
    best = None
    for link in ['ward', 'complete', 'average']:
        cl = AgglomerativeClustering(n_clusters=K, linkage=link).fit_predict(X)
        s = silhouette_score(X, cl)
        if best is None or s > best[1]:
            best = (link, s, cl)
    link, sil, cl = best
    t0 = time.time()
    AgglomerativeClustering(n_clusters=K, linkage=link).fit_predict(X)
    results.append(dict(
        method='Agglomerative', config=f'linkage={link}',
        acc_mean=accuracy_score(y, hungarian_map(y, cl, labs)) * 100,
        acc_sd=0.0, sil=sil, runtime_s=time.time() - t0))

    # --- DBSCAN: grid pada eps dan min_samples (deterministik) ---
    # Rentang eps disesuaikan skala fitur: koordinat ECDF berada di [0,1],
    # sehingga grid 0.1-1.0 dari naskah lama terlalu lebar.
    # Batasan penting: hanya konfigurasi yang menghasilkan tepat K klaster yang
    # dipertimbangkan. Tanpa batasan ini seleksi berbasis silhouette memilih
    # eps sangat kecil yang memecah data menjadi puluhan klaster berisi titik
    # kembar, memberi silhouette = 1.0 yang degenerate. Deskriptor ECDF banyak
    # mengandung nilai kembar sehingga masalah ini pasti muncul.
    span = np.linalg.norm(X.max(axis=0) - X.min(axis=0))
    best = None
    for eps in np.linspace(0.01, 0.40, 40) * span:
        for ms in [5, 10, 20, 30, 50]:
            cl = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
            k_found = len(np.unique(cl[cl >= 0]))
            if k_found != K or (cl < 0).mean() > 0.5:
                continue
            m = cl >= 0
            s = silhouette_score(X[m], cl[m])
            if best is None or s > best[2]:
                best = (eps, ms, s, cl, k_found)
    if best is None:
        results.append(dict(method='DBSCAN', config='tidak ada konfigurasi valid',
                            acc_mean=np.nan, acc_sd=np.nan, sil=np.nan,
                            runtime_s=np.nan))
    else:
        eps, ms, s, cl, k_found = best
        t0 = time.time()
        DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
        results.append(dict(
            method='DBSCAN',
            config=f'eps={eps:.3f}, min_samples={ms}, K ditemukan={k_found}, '
                   f'noise={(cl < 0).mean() * 100:.1f}%',
            acc_mean=accuracy_score(y, hungarian_map(y, cl, labs)) * 100,
            acc_sd=0.0, sil=s, runtime_s=time.time() - t0))

    return pd.DataFrame(results)

## 3. Muat data

Jika sel ini melempar `ValueError`, berkas Excel Anda menyimpan piksel sebagai
repr numpy terpotong dan harus diekstrak ulang dengan
`str(row['R_values'].tolist())`.

In [12]:
imgs, y, filenames = load_pixels(EXCEL)
labs = np.unique(y)
K = len(labs)
print(f"{len(imgs)} citra, ukuran {imgs.shape[1]}x{imgs.shape[2]}, {K} kelas")
print(pd.Series(y).value_counts().to_dict())
print("nama berkas unik:", len(set(filenames)) == len(filenames))

4400 citra, ukuran 32x32, 4 kelas
{'pink': 1100, 'red': 1100, 'white': 1100, 'yellow': 1100}
nama berkas unik: True


## 4. Baseline fitur

Mengisi TODO pertama: colour moments dan histogram RGB terkuantisasi.
Sekaligus menjawab pertanyaan grid tujuh koordinat versus sembilan.

In [13]:
descriptors = {
    "ECDF-7 (diusulkan)":           feat_ecdf7(imgs),
    "ECDF-9 (grid penuh)":          feat_ecdf9(imgs),
    "Colour moments":               feat_colour_moments(imgs),
    f"RGB histogram {HIST_BINS}^3": feat_rgb_histogram(imgs, HIST_BINS),
}

rows = []
for name, F in descriptors.items():
    X = StandardScaler().fit_transform(F) if F.shape[1] > 20 else F
    r, _ = protocol_A(X, y, labs, K, n_seeds=N_SEEDS, tag=name)
    r["d"] = F.shape[1]
    rows.append(r)
    print(f"{name:26s} d={F.shape[1]:5d}  "
          f"acc={r['acc_mean']:.2f}+-{r['acc_sd']:.2f}  ARI={r['ari_mean']:.3f}")

tab_desc = pd.DataFrame(rows)[
    ["representation", "d", "acc_mean", "acc_sd", "acc_min", "acc_max",
     "ari_mean", "nmi_mean", "sil_mean", "db_mean", "ch_mean"]]
tab_desc.to_csv(f"{OUTDIR}/tabel_deskriptor.csv", index=False)
tab_desc.round(3)

ECDF-7 (diusulkan)         d=    7  acc=92.99+-7.34  ARI=0.864
ECDF-9 (grid penuh)        d=    9  acc=89.85+-11.26  ARI=0.840
Colour moments             d=    9  acc=75.21+-12.90  ARI=0.655
RGB histogram 4^3          d=   64  acc=80.02+-13.20  ARI=0.705


,representation,d,acc_mean,acc_sd,acc_min,acc_max,ari_mean,nmi_mean,sil_mean,db_mean,ch_mean
0,ECDF-7 (diusulkan),7,92.991,7.342,70.818,96.864,0.864,0.885,0.709,0.528,11126.274
1,ECDF-9 (grid penuh),9,89.847,11.255,65.591,96.864,0.840,0.884,0.706,0.492,14585.038
2,Colour moments,9,75.211,12.898,65.636,96.864,0.655,0.764,0.712,0.444,15227.528
3,RGB histogram 4^3,64,80.021,13.200,40.636,100.000,0.705,0.802,0.507,1.078,1717.645


## 5. Perbandingan representasi dan uji t berpasangan

In [14]:
F7 = descriptors["ECDF-7 (diusulkan)"]
reps = make_representations(F7)

rows, acc_by_rep = [], {}
for name, X in reps.items():
    r, a = protocol_A(X, y, labs, K, n_seeds=N_SEEDS, tag=name)
    rows.append(r); acc_by_rep[name] = a
    print(f"{name:10s} acc={r['acc_mean']:.2f}+-{r['acc_sd']:.2f}  "
          f"ARI={r['ari_mean']:.3f}  sil={r['sil_mean']:.3f}  DB={r['db_mean']:.3f}")

tab_A = pd.DataFrame(rows)[
    ["representation", "acc_mean", "acc_sd", "acc_ci_lo", "acc_ci_hi",
     "acc_max", "ari_mean", "nmi_mean", "sil_mean", "db_mean", "ch_mean"]]
tab_A.to_csv(f"{OUTDIR}/tabel_protokol_A.csv", index=False)
tab_A.round(3)

raw        acc=92.99+-7.34  ARI=0.864  sil=0.709  DB=0.528
minmax     acc=93.42+-3.47  ARI=0.847  sil=0.711  DB=0.503
zscore     acc=92.48+-4.03  ARI=0.823  sil=0.693  DB=0.542
rank       acc=93.64+-2.73  ARI=0.849  sil=0.671  DB=0.601
quantile   acc=93.91+-0.00  ARI=0.852  sil=0.667  DB=0.610


,representation,acc_mean,acc_sd,acc_ci_lo,acc_ci_hi,acc_max,ari_mean,nmi_mean,sil_mean,db_mean,ch_mean
0,raw,92.991,7.342,71.759,96.864,96.864,0.864,0.885,0.709,0.528,11126.274
1,minmax,93.416,3.474,93.909,93.909,93.909,0.847,0.849,0.711,0.503,9599.715
2,zscore,92.482,4.026,88.136,93.909,93.909,0.823,0.831,0.693,0.542,9059.177
3,rank,93.637,2.725,93.909,93.909,93.909,0.849,0.850,0.671,0.601,8162.272
4,quantile,93.909,0.000,93.909,93.909,93.909,0.852,0.852,0.667,0.610,7960.317


In [15]:
tt = []
for a in reps:
    for b in reps:
        if a >= b:
            continue
        d = acc_by_rep[a] - acc_by_rep[b]
        t, p = stats.ttest_rel(acc_by_rep[a], acc_by_rep[b])
        tt.append(dict(A=a, B=b, mean_diff_pp=d.mean(), t=t, p=p,
                       cohen_dz=d.mean() / d.std(ddof=1)
                                if d.std(ddof=1) > 0 else np.nan))
tab_t = pd.DataFrame(tt)
tab_t.to_csv(f"{OUTDIR}/tabel_uji_t.csv", index=False)
tab_t.round(4)

,A,B,mean_diff_pp,t,p,cohen_dz
0,raw,zscore,0.5084,0.7301,0.4670,0.0730
1,minmax,raw,0.4257,0.5682,0.5712,0.0568
2,minmax,zscore,0.9341,2.3437,0.0211,0.2344
3,minmax,rank,-0.2202,-0.4954,0.6214,-0.0495
4,minmax,quantile,-0.4927,-1.4185,0.1592,-0.1418
5,rank,raw,0.6459,0.8214,0.4134,0.0821
6,rank,zscore,1.1543,2.3361,0.0215,0.2336
7,quantile,raw,0.9184,1.2509,0.2139,0.1251
8,quantile,zscore,1.4268,3.5444,0.0006,0.3544
9,quantile,rank,0.2725,1.0000,0.3197,0.1000


## 6. Protokol B

Cross-validation lima lipatan, 20 seed. Transformasi di-fit hanya pada fold
latih, sehingga transformasi rank tidak membocorkan informasi dari fold uji.
Sekitar tiga menit.

In [16]:
rows = [protocol_B(F7, y, labs, K, transform=t)
        for t in ["raw", "minmax", "zscore", "rank", "quantile"]]
tab_B = pd.DataFrame(rows)
tab_B.to_csv(f"{OUTDIR}/tabel_protokol_B.csv", index=False)
tab_B.round(3)

,transform,n_runs,acc_mean,acc_sd,acc_ci_lo,acc_ci_hi
0,raw,100,93.648,6.269,70.963,97.500
1,minmax,100,92.486,5.708,70.963,94.432
2,zscore,100,91.615,4.822,72.534,94.432
3,rank,100,92.817,5.400,69.651,94.432
4,quantile,100,93.403,3.668,93.295,94.432


## 7. Grid search baseline clustering

Mengisi TODO ketiga. Hyperparameter dipilih dengan silhouette, bukan akurasi,
agar label tidak bocor ke baseline.

In [17]:
tab_gs = grid_search_baselines(F7, y, labs, K, n_seeds=30)
tab_gs.to_csv(f"{OUTDIR}/tabel_grid_search.csv", index=False)
tab_gs.round(3)

,method,config,acc_mean,acc_sd,sil,runtime_s
0,K-Means (k-means++),K=4,93.326,6.232,0.708,0.238
1,Gaussian mixture (kmeans),cov=full,93.326,6.232,0.708,0.247
2,Gaussian mixture (random_from_data),cov=spherical,75.709,12.709,0.625,0.226
3,Agglomerative,linkage=ward,95.841,0.000,0.710,0.191
4,DBSCAN,"eps=0.310, min_samples=50, K ditemukan=4, nois...",72.773,0.000,0.610,0.090


## 8. ResNet-50 (opsional)

Perlu `torch`, `torchvision`, `Pillow`. Jika belum terpasang, jalankan sel
instalasi di bawah sekali, lalu **restart kernel** dan ulangi dari sel 1.

Dijalankan dua kali:

* **8a** memakai citra asli dari `IMAGE_ROOT`. Ini perbandingan yang jujur
  untuk ResNet dan yang akan diminta reviewer.
* **8b** memakai citra 32x32 hasil rekonstruksi yang di-upsample. Ini
  perbandingan input-identik dengan ECDF-7.

Laporkan keduanya di naskah. Tanpa GPU, tiap tahap memakan 10--20 menit.

In [18]:
# Hapus tanda pagar, jalankan sekali, lalu restart kernel:
#%pip install torch torchvision pillow


In [19]:
# --- 8a. citra asli ---
F_resnet_orig = feat_resnet50(image_root=IMAGE_ROOT, filenames=filenames)
print("dimensi fitur:", F_resnet_orig.shape[1])

X = StandardScaler().fit_transform(F_resnet_orig)
r_orig, _ = protocol_A(X, y, labs, K, n_seeds=N_SEEDS,
                       tag="ResNet-50 (citra asli)")
print(f"acc={r_orig['acc_mean']:.2f}+-{r_orig['acc_sd']:.2f}  "
      f"ARI={r_orig['ari_mean']:.3f}")

4400 citra terindeks dari C:/Users/Ahsan/Downloads/data
dimensi fitur: 2048
acc=48.52+-12.75  ARI=0.259


In [20]:
# --- 8b. citra 32x32 di-upsample (input identik dengan ECDF-7) ---
F_resnet_32 = feat_resnet50(imgs=imgs)
X = StandardScaler().fit_transform(F_resnet_32)
r_32, _ = protocol_A(X, y, labs, K, n_seeds=N_SEEDS,
                     tag="ResNet-50 (32x32 upsampled)")
print(f"acc={r_32['acc_mean']:.2f}+-{r_32['acc_sd']:.2f}  "
      f"ARI={r_32['ari_mean']:.3f}")

tab_resnet = pd.DataFrame([r_orig, r_32])
tab_resnet.to_csv(f"{OUTDIR}/tabel_resnet.csv", index=False)
tab_resnet[["representation", "acc_mean", "acc_sd", "ari_mean",
            "sil_mean", "runtime_s"]].round(3)

acc=60.92+-11.52  ARI=0.400


,representation,acc_mean,acc_sd,ari_mean,sil_mean,runtime_s
0,ResNet-50 (citra asli),48.523,12.751,0.259,0.370,34.15
1,ResNet-50 (32x32 upsampled),60.919,11.521,0.400,0.327,33.76


## 9. Ekspor ke LaTeX

Keluaran sel ini bisa langsung ditempel ke `sn-article.tex`.

In [21]:
def to_latex(df, caption, label, cols=None, digits=2):
    d = df[cols] if cols else df
    print(d.to_latex(index=False, float_format=f"%.{digits}f",
                     caption=caption, label=label,
                     column_format="@{}l" + "c" * (d.shape[1] - 1) + "@{}"))

to_latex(tab_desc, "Perbandingan deskriptor, Protokol A.", "tab:descriptors",
         ["representation", "d", "acc_mean", "acc_sd", "ari_mean", "sil_mean"])

\begin{table}
\centering
\caption{Perbandingan deskriptor, Protokol A.}
\label{tab:descriptors}
\begin{tabular}{@{}lccccc@{}}
\toprule
     representation &  d &  acc\_mean &  acc\_sd &  ari\_mean &  sil\_mean \\
\midrule
 ECDF-7 (diusulkan) &  7 &     92.99 &    7.34 &      0.86 &      0.71 \\
ECDF-9 (grid penuh) &  9 &     89.85 &   11.26 &      0.84 &      0.71 \\
     Colour moments &  9 &     75.21 &   12.90 &      0.65 &      0.71 \\
  RGB histogram 4\textasciicircum 3 & 64 &     80.02 &   13.20 &      0.70 &      0.51 \\
\bottomrule
\end{tabular}
\end{table}



In [22]:
to_latex(tab_gs, "Baseline clustering dengan grid search.", "tab:clustering",
         ["method", "config", "acc_mean", "acc_sd", "sil", "runtime_s"], digits=3)

\begin{table}
\centering
\caption{Baseline clustering dengan grid search.}
\label{tab:clustering}
\begin{tabular}{@{}lccccc@{}}
\toprule
                             method &                                             config &  acc\_mean &  acc\_sd &   sil &  runtime\_s \\
\midrule
                K-Means (k-means++) &                                                K=4 &    93.326 &   6.232 & 0.708 &      0.238 \\
          Gaussian mixture (kmeans) &                                           cov=full &    93.326 &   6.232 & 0.708 &      0.247 \\
Gaussian mixture (random\_from\_data) &                                      cov=spherical &    75.709 &  12.709 & 0.625 &      0.226 \\
                      Agglomerative &                                       linkage=ward &    95.841 &   0.000 & 0.710 &      0.191 \\
                             DBSCAN & eps=0.310, min\_samples=50, K ditemukan=4, noise... &    72.773 &   0.000 & 0.610 &      0.090 \\
\bottomrule
\end{tabular}
\end{table}


In [23]:
to_latex(tab_B, "Protokol B: cross-validation bebas kebocoran.", "tab:protocolB",
         ["transform", "acc_mean", "acc_sd", "acc_ci_lo", "acc_ci_hi"])

\begin{table}
\centering
\caption{Protokol B: cross-validation bebas kebocoran.}
\label{tab:protocolB}
\begin{tabular}{@{}lcccc@{}}
\toprule
transform &  acc\_mean &  acc\_sd &  acc\_ci\_lo &  acc\_ci\_hi \\
\midrule
      raw &     93.65 &    6.27 &      70.96 &      97.50 \\
   minmax &     92.49 &    5.71 &      70.96 &      94.43 \\
   zscore &     91.61 &    4.82 &      72.53 &      94.43 \\
     rank &     92.82 &    5.40 &      69.65 &      94.43 \\
 quantile &     93.40 &    3.67 &      93.30 &      94.43 \\
\bottomrule
\end{tabular}
\end{table}



In [24]:
from PIL import Image
import os, numpy as np, pandas as pd

ROOT = IMAGE_ROOT          # folder induk berisi pink_rose, red_rose, white_rose, yellow_rose
LEVELS_7 = [("R",100),("R",150),("R",200),("G",150),("G",200),("B",150),("B",200)]

def descriptor_at(root, p):
    """Baca citra asli, resize ke p x p, kembalikan deskriptor ECDF-7 dan label."""
    X, lab = [], []
    for folder in sorted(os.listdir(root)):
        d = os.path.join(root, folder)
        if not os.path.isdir(d):
            continue
        for fn in sorted(os.listdir(d)):
            if not fn.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            a = np.asarray(Image.open(os.path.join(d, fn))
                           .convert("RGB").resize((p, p)), dtype=np.int16)
            ch = {"R": a[:,:,0].ravel(), "G": a[:,:,1].ravel(), "B": a[:,:,2].ravel()}
            X.append([(ch[c] <= t).mean() for c, t in LEVELS_7])
            lab.append(folder)
    return np.array(X), np.array(lab)

rows = []
for p in [16, 32, 64, 128]:
    Xp, yp = descriptor_at(ROOT, p)
    labs_p, Kp = np.unique(yp), len(np.unique(yp))
    r, _ = protocol_A(Xp, yp, labs_p, Kp, n_seeds=N_SEEDS, tag=f"p={p}")
    r["p"] = p
    rows.append(r)
    print(f"p={p:3d}  n={len(Xp)}  acc={r['acc_mean']:.2f}+-{r['acc_sd']:.2f}  "
          f"ARI={r['ari_mean']:.3f}  sil={r['sil_mean']:.3f}")

tab_p = pd.DataFrame(rows)[["p","acc_mean","acc_sd","acc_max","ari_mean","sil_mean"]]
tab_p.to_csv(f"{OUTDIR}/tabel_ablasi_ukuran.csv", index=False)
tab_p.round(3)

p= 16  n=4400  acc=91.30+-7.23  ARI=0.823  sil=0.715
p= 32  n=4400  acc=90.78+-7.77  ARI=0.821  sil=0.705
p= 64  n=4400  acc=89.42+-9.18  ARI=0.812  sil=0.703
p=128  n=4400  acc=92.11+-8.63  ARI=0.857  sil=0.706


,p,acc_mean,acc_sd,acc_max,ari_mean,sil_mean
0,16,91.299,7.235,93.909,0.823,0.715
1,32,90.782,7.766,93.909,0.821,0.705
2,64,89.417,9.182,98.068,0.812,0.703
3,128,92.114,8.627,96.864,0.857,0.706


---# 10. Flowers102: unduh, kategorikan, ekstrakSel-sel berikut mengulang pipeline Flowers102 Anda dari nol: unduh dataset dariOxford VGG, kategorikan ke lima warna, ekstrak deskriptor, dan tulis Excel yangbisa langsung dipakai sel 3.**Tiga perbedaan dari skrip Anda yang lama, semuanya perbaikan bug:**1. `str(R.tolist())` menggantikan `str(R)`. Ini penyebab berkas lama tersimpan   terpotong (`[82 85 80 ... 92 94 95]`) dan piksel aslinya hilang.2. Kolom `label` diambil dari kategori warna, bukan dari nama folder induk.3. Batas `max_images_to_process` dinaikkan agar kelas `brown` punya peluang   mencapai kuota; di skrip lama ia berhenti di 21 citra.**Peringatan metodologis.** Label warna di sini **dihasilkan otomatis**, bukandianotasi manusia: warna dominan dicari dengan K-means lalu diklasifikasidengan ambang HSV. Ini harus dinyatakan terbuka di naskah. Lihat catatan disel terakhir bagian ini.Unduhan sekitar 330 MB dan hanya perlu dijalankan sekali.

## 10.1 Unduh dan ekstrak arsip

In [ ]:
import os, tarfile, shutilfrom pathlib import Pathimport requestsimport cv2import numpy as npimport pandas as pdfrom tqdm.auto import tqdmFLOWERS_DIR   = Path("./data/flowers102")COLOR_DIR     = Path("./data/flowers102_5colors")FLOWERS_EXCEL = "data_ekstraksi_rgb_flowers102_final.xlsx"P_FLOWERS     = 32URL_IMAGES = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz"def unduh(url, dest):    dest = Path(dest)    if dest.exists() and dest.stat().st_size > 0:        print("sudah ada:", dest)        return dest    dest.parent.mkdir(parents=True, exist_ok=True)    tmp = dest.with_suffix(dest.suffix + ".part")    r = requests.get(url, stream=True, timeout=60)    r.raise_for_status()    total = int(r.headers.get("content-length", 0))    with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,                                    desc=dest.name) as bar:        for chunk in r.iter_content(chunk_size=1 << 16):            bar.update(f.write(chunk))    tmp.rename(dest)          # tulis ke .part dulu agar unduhan putus tidak    return dest               # meninggalkan arsip rusak yang tampak lengkaparsip = unduh(URL_IMAGES, FLOWERS_DIR / "102flowers.tgz")images_dir = FLOWERS_DIR / "jpg"if not images_dir.exists():    print("mengekstrak arsip ...")    with tarfile.open(arsip, "r:gz") as tar:        tar.extractall(FLOWERS_DIR)n = len(list(images_dir.glob("*.jpg")))print(f"{n} citra tersedia di {images_dir}")assert n == 8189, f"jumlah citra tidak wajar ({n}), arsip mungkin rusak"

## 10.2 Kategorikan ke lima warnaLogikanya identik dengan skrip Anda supaya labelnya reproducible: warna dominanlewat K-means, lalu aturan HSV dengan fallback rasio RGB.

In [ ]:
from sklearn.cluster import KMeansCOLOR_CATEGORIES = ["red", "blue", "yellow", "purple", "brown"]def warna_dominan(img_path, n_clusters=3):    img = cv2.imread(str(img_path))    if img is None:        return None    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    piksel = cv2.resize(img, (100, 100)).reshape(-1, 3)    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit(piksel)    return km.cluster_centers_[np.argmax(np.bincount(km.labels_))]def kategorikan(img_path):    c = warna_dominan(img_path)    if c is None:        return "unknown"    R, G, B = c    H, S, V = cv2.cvtColor(np.uint8([[c]]), cv2.COLOR_RGB2HSV)[0][0]    if (H <= 10 or H >= 170) and S > 80:      return "red"    elif 100 <= H <= 140 and S > 60:          return "blue"    elif 20 <= H <= 40 and S > 60:            return "yellow"    elif 130 <= H <= 170 and S > 50:          return "purple"    elif 10 <= H <= 20 and 40 <= S <= 80 and V < 180: return "brown"    total = R + G + B + 1e-3    r, g, b = R / total, G / total, B / total    if   r > 0.5 and g < 0.3 and b < 0.3:     return "red"    elif b > 0.4 and r < 0.3:                 return "blue"    elif r > 0.4 and g > 0.4 and b < 0.2:     return "yellow"    elif r > 0.3 and b > 0.3 and g < 0.3:     return "purple"    elif 0.2 < r < 0.4 and 0.2 < g < 0.4 and b < 0.2: return "brown"    return "mixed"def bangun_dataset_warna(images_dir, per_kelas=40, maks_citra=8189):    """Salin citra ke folder per warna sampai tiap kelas mencapai kuota."""    for c in COLOR_CATEGORIES:        (COLOR_DIR / c).mkdir(parents=True, exist_ok=True)    jumlah = {c: len(list((COLOR_DIR / c).glob("*.jpg"))) for c in COLOR_CATEGORIES}    if all(v >= per_kelas for v in jumlah.values()):        print("folder warna sudah lengkap:", jumlah)        return jumlah    berkas = sorted(images_dir.glob("*.jpg"))[:maks_citra]    for p in tqdm(berkas, desc="mengategorikan"):        if all(v >= per_kelas for v in jumlah.values()):            break        k = kategorikan(p)        if k in COLOR_CATEGORIES and jumlah[k] < per_kelas:            shutil.copy(p, COLOR_DIR / k / p.name)            jumlah[k] += 1    return jumlah# maks_citra dinaikkan ke seluruh dataset; skrip lama membatasinya di 3000# sehingga kelas brown berhenti di 21 citra.jumlah = bangun_dataset_warna(images_dir, per_kelas=40, maks_citra=8189)print(jumlah)for c, v in jumlah.items():    if v < 40:        print(f"  catatan: kelas {c} hanya {v} citra; aturan HSV-nya sempit "              f"sehingga kuota tidak tercapai walau seluruh 8189 citra dipindai")

## 10.3 Ekstrak dan tulis Excel

In [ ]:
def ekstrak_folder_warna(color_dir, p=P_FLOWERS):    """Pipeline identik dengan notebook mawar: cv2, bilinear, BGR lalu RGB."""    baris = []    for warna in COLOR_CATEGORIES:        folder = Path(color_dir) / warna        berkas = sorted(folder.glob("*.jpg"))        print(f"{warna:7s}: {len(berkas)} citra")        for i, fp in enumerate(berkas, start=1):            img = cv2.imread(str(fp))            if img is None:                print("  gagal dibaca, dilewati:", fp.name)                continue            img = cv2.resize(img, (p, p))                    # INTER_LINEAR            a = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            R, G, B = a[:, :, 0].ravel(), a[:, :, 1].ravel(), a[:, :, 2].ravel()            baris.append(dict(                filename=fp.name, gambar_ke=i, label=warna,                p=p, total_pixels=p * p,                R_values=str(R.tolist()),   # .tolist() -- tanpa ini data terpotong                G_values=str(G.tolist()),                B_values=str(B.tolist()),                avg_R=R.mean(), avg_G=G.mean(), avg_B=B.mean(),                min_R=R.min(), max_R=R.max(), min_G=G.min(),                max_G=G.max(), min_B=B.min(), max_B=B.max()))    return pd.DataFrame(baris)df_fl = ekstrak_folder_warna(COLOR_DIR)# uji penerimaan sebelum menyimpans = str(df_fl["R_values"].iloc[0])assert "..." not in s, "masih terpotong -- periksa .tolist()"assert len(s) > 3000, f"string terlalu pendek ({len(s)}), data tidak utuh"assert df_fl["filename"].nunique() == len(df_fl), "ada nama berkas kembar"assert set(df_fl["label"]) <= set(COLOR_CATEGORIES), "ada label tak dikenal"df_fl.to_excel(FLOWERS_EXCEL, index=False)print(f"\ntersimpan: {FLOWERS_EXCEL}")print(f"{len(df_fl)} baris | panjang string piksel: {len(s)}")print(df_fl["label"].value_counts().to_dict())

## 10.4 Jalankan analisis pada Flowers102Setelah Excel jadi, jalankan seluruh protokol pada dataset ini. Semua fungsisudah generik terhadap jumlah kelas, jadi $K=5$ terdeteksi otomatis.

In [ ]:
imgs_fl, y_fl, fn_fl = load_pixels(FLOWERS_EXCEL)labs_fl, K_fl = np.unique(y_fl), len(np.unique(y_fl))print(f"{len(imgs_fl)} citra, {K_fl} kelas: {pd.Series(y_fl).value_counts().to_dict()}")F7_fl = feat_ecdf7(imgs_fl)desc_fl = {    "ECDF-7":            F7_fl,    "ECDF-9":            feat_ecdf9(imgs_fl),    "Colour moments":    feat_colour_moments(imgs_fl),    "RGB histogram 4^3": feat_rgb_histogram(imgs_fl, HIST_BINS),}rows = []for nama, Fx in desc_fl.items():    X = StandardScaler().fit_transform(Fx) if Fx.shape[1] > 20 else Fx    r, _ = protocol_A(X, y_fl, labs_fl, K_fl, n_seeds=N_SEEDS, tag=nama)    r["d"] = Fx.shape[1]; rows.append(r)    print(f"{nama:18s} d={Fx.shape[1]:4d} acc={r['acc_mean']:6.2f}+-{r['acc_sd']:5.2f} "          f"ARI={r['ari_mean']:.3f} NMI={r['nmi_mean']:.3f}")tab_fl = pd.DataFrame(rows)tab_fl.to_csv(f"{OUTDIR}/flowers102_tabel_deskriptor.csv", index=False)tab_fl[["representation", "d", "acc_mean", "acc_sd", "ari_mean",        "nmi_mean", "sil_mean"]].round(3)

In [ ]:
# batas atas tersupervisi: apakah keterbatasannya informasional atau algoritmik?from sklearn.svm import LinearSVCfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDAfrom sklearn.model_selection import cross_val_scoreprint("Linear SVM:", round(cross_val_score(LinearSVC(max_iter=20000),                                           F7_fl, y_fl, cv=5).mean() * 100, 2), "%")print("LDA       :", round(cross_val_score(LDA(), F7_fl, y_fl, cv=5).mean() * 100, 2), "%")# batas Chebyshev per pasangan kelasnames = ["R_100","R_150","R_200","G_150","G_200","B_150","B_200"]out = []for i in range(K_fl):    for j in range(i + 1, K_fl):        a, b = labs_fl[i], labs_fl[j]; best = None        for k in range(7):            d = abs(F7_fl[y_fl == a, k].mean() - F7_fl[y_fl == b, k].mean())            s2 = max(F7_fl[y_fl == a, k].var(), F7_fl[y_fl == b, k].var())            e = 4 * s2 / d**2 if d > 0 else np.inf            if best is None or e < best[0]:                best = (e, names[k], d, np.sqrt(s2))        e, nm, d, sd = best        out.append(dict(pair=f"{a} vs {b}", coord=nm, delta=d, sigma=sd,                        eps_pct=min(e, 1.0) * 100))tab_cheb_fl = pd.DataFrame(out).sort_values("eps_pct")tab_cheb_fl.to_csv(f"{OUTDIR}/flowers102_chebyshev.csv", index=False)# rasio varians antar-kelas / dalam-kelasA = imgs_fl.reshape(len(imgs_fl), -1, 3).astype(float).mean(axis=1)gm = A.mean(axis=0)sb = sum(len(A[y_fl == l]) * np.sum((A[y_fl == l].mean(axis=0) - gm) ** 2) for l in labs_fl)sw = sum(np.sum((A[y_fl == l] - A[y_fl == l].mean(axis=0)) ** 2) for l in labs_fl)print(f"\nB/W Flowers102 = {sb/sw:.4f}")tab_cheb_fl.round(3)

In [ ]:
# Diagnosis aturan pelabelan: uji pada warna murni, tanpa perlu dataset.# Jalankan ini dan sertakan keluarannya sebagai lampiran naskah.uji = [("merah murni",   [255, 0, 0]),       ("biru murni",    [0, 0, 255]),       ("kuning murni",  [255, 255, 0]),       ("ungu murni",    [128, 0, 255]),       ("ungu tua",      [102, 0, 153]),       ("coklat khas",   [139, 90, 43]),       ("abu netral",    [128, 128, 128])]print("warna murni -> kategori yang diberikan aturan:")for nama, c in uji:    H, S, V = cv2.cvtColor(np.uint8([[c]]), cv2.COLOR_RGB2HSV)[0][0]    hasil = kategorikan.__wrapped__(c) if hasattr(kategorikan, "__wrapped__") else None    # panggil ulang logikanya langsung dari warna, tanpa membaca berkas    R, G, B = c    if   (H <= 10 or H >= 170) and S > 80:            k = "red"    elif 100 <= H <= 140 and S > 60:                  k = "blue"    elif 20 <= H <= 40 and S > 60:                    k = "yellow"    elif 130 <= H <= 170 and S > 50:                  k = "purple"    elif 10 <= H <= 20 and 40 <= S <= 80 and V < 180: k = "brown"    else:        t = R + G + B + 1e-3; r, g, b = R/t, G/t, B/t        if   r > 0.5 and g < 0.3 and b < 0.3:          k = "red"        elif b > 0.4 and r < 0.3:                      k = "blue"        elif r > 0.4 and g > 0.4 and b < 0.2:          k = "yellow"        elif r > 0.3 and b > 0.3 and g < 0.3:          k = "purple"        elif 0.2 < r < 0.4 and 0.2 < g < 0.4 and b < 0.2: k = "brown"        else:                                          k = "mixed"    tanda = "  <-- salah" if (nama.startswith("ungu") and k != "purple") or \                             (nama.startswith("coklat") and k != "brown") else ""    print(f"  {nama:13s} RGB={str(c):16s} HSV=({H:3d},{S:3d},{V:3d}) -> {k}{tanda}")print("\nDua kegagalan yang terlihat di atas:")print("  1. Ungu murni jatuh ke 'blue'. Rentang hue blue (100-140) dievaluasi")print("     sebelum purple (130-170), sehingga irisan 130-140 selalu jadi blue.")print("  2. Coklat khas jatuh ke 'mixed'. Syarat brown menuntut H 10-20,")print("     S 40-80, dan V<180 sekaligus, sehingga hampir tidak pernah terpenuhi.")print("     Inilah sebabnya kelas brown hanya terkumpul 21 dari kuota 40.")

## 10.5 Yang harus dinyatakan di naskahLabel warna di sini dihasilkan oleh `kategorikan()`, bukan oleh anotatormanusia. Prosedurnya: warna dominan dicari dengan K-means tiga klaster padacitra 100×100, lalu diklasifikasi dengan ambang HSV, dengan fallback rasio RGBbila tidak ada aturan yang cocok.Ini wajib diungkap, dan ada tiga konsekuensi yang reviewer akan cari.**Sirkularitas.** Label diturunkan dari warna, lalu diklaster berdasarkanwarna. Kalau metodenya berhasil, sebagian keberhasilan itu bawaan konstruksi.Namun pada dataset ini metodenya justru *gagal*, sehingga kegagalannya menjaditemuan yang lebih kuat: deskriptor global tidak dapat memulihkan kategori yangdibuat oleh heuristik warna dominan lokal. Dua besaran ini mengukur hal yangberbeda — warna dominan satu klaster versus distribusi seluruh citra.**Aturan yang bertumpang tindih.** Rentang `purple` (H 130--170) beririsandengan `blue` (H 100--140) dan `red` (H ≥ 170). Karena dievaluasi berurutan,citra dengan H antara 130 dan 140 selalu jatuh ke `blue`. Batas kelasnyakarena itu artefak urutan `if`, bukan batas persepsi warna.**Ketimpangan kelas.** Kelas `brown` memerlukan H 10--20, S 40--80, dan V < 180sekaligus, sehingga jarang terpenuhi. Di ekstraksi lama ia berhenti di 21 citradari kuota 40.Kalimat yang saya sarankan untuk Section~4.1, menggantikan klaim anotasimanual:> The colour labels of Flowers102-C are not human annotations. They were> generated automatically: the dominant colour of each image was obtained by> $K$-means with three clusters on a $100\times100$ version of the image, then> mapped to one of five categories by thresholds on hue and saturation, with a> fallback rule on RGB ratios. The category boundaries are therefore artefacts> of those thresholds rather than perceptual divisions, and the hue ranges for> blue and purple overlap, so images with hue between 130 and 140 are always> assigned to blue. We report this because it bears on the interpretation of> Section~5.1: the labels are themselves a function of colour, so the failure> of a colour descriptor to recover them is a statement about the mismatch> between a global distributional representation and a local dominant-colour> heuristic, not about colour information being absent.

---# 11. Rose-4400: unduh dari Kaggle dan ekstrak ulangMembuat jalur reproduksi penuh untuk dataset mawar, dari unduhan sampai Excel.Reviewer yang ingin mengulang eksperimen tidak perlu meminta data Anda.**Autentikasi.** `kagglehub` memerlukan kredensial Kaggle. Kalau belum pernahdisiapkan, unduh `kaggle.json` dari halaman Account di kaggle.com lalu simpandi `C:\Users\<nama>\.kaggle\kaggle.json`. Tanpa itu unduhan gagal dengan galat`401 Unauthorized`.**Peringatan reproduksi.** Dataset di Kaggle bisa saja berbeda dari salinanyang Anda pakai membuat `data_ekstraksi_rgb_4kelas.xlsx`: penulisnya dapatmemperbarui isinya, dan nama folder atau jumlah citra per kelas mungkin tidaksama. Sel 11.4 membandingkan hasil ekstraksi baru dengan Excel lama danmemberi tahu Anda apakah keduanya identik. **Jangan mengganti angka di naskahdengan hasil ekstraksi baru sebelum uji itu lolos.**

## 11.1 Unduh dataset

In [ ]:
# Jalankan sekali kalau kagglehub belum terpasang:# %pip install kagglehubimport kagglehubpath = kagglehub.dataset_download(    "shuvokumarbasak2030/rose-color-classification-new-and-update-dataset")print("Path to dataset files:", path)

## 11.2 Periksa struktur folderJangan langsung mengekstrak. Lihat dulu bentuk folder dan jumlah citranya,karena nama folder di Kaggle belum tentu sama dengan `pink_rose`, `red_rose`,`white_rose`, `yellow_rose` yang Anda pakai sebelumnya.

In [ ]:
import osfrom pathlib import Pathfrom collections import CounterROOT_KAGGLE = Path(path)EKSTENSI = (".jpg", ".jpeg", ".png")isi = Counter()for dirpath, _, files in os.walk(ROOT_KAGGLE):    n = sum(1 for f in files if f.lower().endswith(EKSTENSI))    if n:        isi[os.path.relpath(dirpath, ROOT_KAGGLE)] = nprint(f"total citra: {sum(isi.values())}\n")for folder, n in sorted(isi.items()):    print(f"  {n:6d}  {folder}")

## 11.3 Petakan folder ke label, lalu ekstrakSesuaikan `PETA_LABEL` dengan keluaran sel di atas. Kunci adalah path relatiffolder, nilai adalah label kelas yang dipakai naskah.Kalau dataset punya pembagian train/test, gabungkan keduanya: eksperimen iniadalah clustering, jadi tidak ada pemisahan latih-uji di tingkat data.

In [ ]:
import cv2import numpy as npimport pandas as pdfrom tqdm.auto import tqdmP_ROSE       = 32ROSE_EXCEL   = "data_ekstraksi_rgb_4kelas_baru.xlsx"# SESUAIKAN dengan keluaran sel 11.2. Contoh untuk struktur datar:PETA_LABEL = {    "pink_rose":   "pink",    "red_rose":    "red",    "white_rose":  "white",    "yellow_rose": "yellow",}# Contoh kalau ada subfolder train/test:# PETA_LABEL = {#     "train/pink_rose": "pink", "test/pink_rose": "pink",#     "train/red_rose":  "red",  "test/red_rose":  "red",#     ...# }tidak_terpetakan = [f for f in isi if f not in PETA_LABEL]if tidak_terpetakan:    print("folder berisi citra yang belum dipetakan (akan dilewati):")    for f in tidak_terpetakan:        print("   ", f, f"({isi[f]} citra)")def ekstrak_mawar(root, peta, p=P_ROSE):    """Pipeline identik dengan notebook ekstraksi asli Anda:    cv2.imread (BGR) -> cv2.resize bilinear -> cvtColor ke RGB."""    baris, hitung = [], Counter()    for rel, label in peta.items():        folder = Path(root) / rel        if not folder.is_dir():            print("folder tidak ada, dilewati:", rel)            continue        berkas = sorted(f for f in folder.iterdir()                        if f.suffix.lower() in EKSTENSI)        for fp in tqdm(berkas, desc=f"{label:7s}", leave=False):            img = cv2.imread(str(fp))            if img is None:                print("  gagal dibaca, dilewati:", fp.name)                continue            img = cv2.resize(img, (p, p))                   # INTER_LINEAR            a = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            R, G, B = a[:, :, 0].ravel(), a[:, :, 1].ravel(), a[:, :, 2].ravel()            hitung[label] += 1            baris.append(dict(                filename=fp.name, gambar_ke=hitung[label], label=label,                p=p, total_pixels=p * p,                R_values=str(R.tolist()),   # .tolist() wajib, jangan str(R)                G_values=str(G.tolist()),                B_values=str(B.tolist()),                avg_R=R.mean(), avg_G=G.mean(), avg_B=B.mean(),                min_R=R.min(), max_R=R.max(), min_G=G.min(),                max_G=G.max(), min_B=B.min(), max_B=B.max()))    return pd.DataFrame(baris)df_rose = ekstrak_mawar(ROOT_KAGGLE, PETA_LABEL)s = str(df_rose["R_values"].iloc[0])assert "..." not in s and len(s) > 3000, "data piksel terpotong, periksa .tolist()"if df_rose["filename"].nunique() != len(df_rose):    print("PERINGATAN: ada nama berkas kembar antar folder. Pemetaan citra ke "          "label masih benar karena label diambil saat penelusuran, tetapi "          "pencocokan dengan Excel lama di sel 11.4 bisa keliru.")df_rose.to_excel(ROSE_EXCEL, index=False)print(f"\ntersimpan: {ROSE_EXCEL}")print(f"{len(df_rose)} baris | {df_rose['label'].value_counts().to_dict()}")

## 11.4 Uji kesetaraan dengan ekstraksi lamaIni yang menentukan apakah angka di naskah boleh diganti. Tiga pemeriksaan:jumlah citra per kelas, irisan nama berkas, dan kesamaan deskriptor ECDF-7 padacitra yang sama.

In [ ]:
# Uji kesetaraan berbasis ISI, bukan nama berkas.# Nama berkas tidak bisa dipakai: penamaan lokal Anda (pink (1).jpg) berbeda# dari penamaan di arsip Kaggle, meski citranya sama.import numpy as np, pandas as pdimgs_b, y_b, fn_b = load_pixels(ROSE_EXCEL)     # ekstraksi KaggleF_b = feat_ecdf7(imgs_b)F_l = F7                                         # ekstraksi lama, dari sel 5print(f"lama {F_l.shape} | baru {F_b.shape}\n")identik_semua = Truefor lab in np.unique(y):    A, B = F_l[y == lab], F_b[y_b == lab]    if len(A) != len(B):        print(f"{lab:7s}: jumlah beda ({len(A)} vs {len(B)})")        identik_semua = False        continue    # urutkan baris leksikografis di kedua sisi agar urutan berkas tidak relevan    ia, ib = np.lexsort(A.T[::-1]), np.lexsort(B.T[::-1])    beda = np.abs(A[ia] - B[ib])    sama = beda.max() < 1e-9    identik_semua &= sama    print(f"{lab:7s}: n={len(A):5d}  selisih maks {beda.max():.3e}  "          f"{'IDENTIK' if sama else 'berbeda'}")print()if identik_semua:    print("Himpunan citra kedua ekstraksi IDENTIK; hanya nama berkasnya berbeda.")    print("Sumber Kaggle boleh disebut sebagai jalur reproduksi penuh di naskah.")else:    from scipy.spatial import cKDTree    d_all = []    for lab in np.unique(y):        A, B = F_l[y == lab], F_b[y_b == lab]        if len(A) and len(B):            d, _ = cKDTree(B).query(A, k=1)            d_all.append(d)            print(f"{lab:7s}: jarak ke padanan terdekat -- median {np.median(d):.4f}, "                  f"p95 {np.percentile(d, 95):.4f}, maks {d.max():.4f}")    d = np.concatenate(d_all)    print(f"\n{(d < 1e-6).mean()*100:.1f}% citra punya padanan persis, "          f"{(d < 0.01).mean()*100:.1f}% padanan sangat dekat.")    print("Laporkan angka ini di Data availability; jangan mencampur angka dari "          "dua ekstraksi berbeda dalam satu tabel.")

In [ ]:
# Kalau uji di atas lolos, jalankan ini untuk memastikan Protokol A pada# ekstraksi baru mereproduksi angka Tabel 3 (92.99 +- 7.34).imgs_b, y_b, fn_b = load_pixels(ROSE_EXCEL)labs_b = np.unique(y_b)r_b, _ = protocol_A(feat_ecdf7(imgs_b), y_b, labs_b, len(labs_b),                    n_seeds=N_SEEDS, tag="ECDF-7 (ekstraksi Kaggle)")print(f"acc={r_b['acc_mean']:.2f}+-{r_b['acc_sd']:.2f}  ARI={r_b['ari_mean']:.3f}")selisih = abs(r_b["acc_mean"] - 92.99)print(f"selisih dari Tabel 3: {selisih:.2f} poin",      "-- LOLOS" if selisih < 0.5 else "-- PERIKSA LAGI")